# Minimal cross-store query — Netherlands 2024

Pull **CO2** from the ICOS Obspack zarr and **monthly NEE** from the
FLUXNET zarr for stations inside a Netherlands bounding box, restricted
to 2024.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

Each store has a *combined-view group* (built by `combine_to_dim.py`)
where every station shares one `(station, time)` axis pair, so spatial
and temporal selection becomes a single xarray expression — no
per-group loops, no `.zattrs` parsing.

| Store | Combined group | Spatial coords |
|---|---|---|
| `icos-obspack.zarr` | `co2`, `ch4`, `n2o`, `co` | `lat`, `lon`, `intake_height` |
| `icos-fluxnet.zarr` | `_combined/fluxnet_dd` … `_yy` | `lat`, `lon`, `station_elevation` |

In [5]:
import xarray as xr

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

## Obspack — CO2

In [6]:
ds = xr.open_zarr("icos-obspack.zarr", group="co2", consolidated=False)

co2_nl = (
    ds["co2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time_co2=slice(T0, T1))
)

# Per-station summary — coords travel with the DataArray
for sid in co2_nl.station.values:
    da = co2_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(co2_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(co2_nl.lon.sel(station=sid)):.3f}  "
          f"intake_height={float(co2_nl.intake_height.sel(station=sid)):>4.0f} m  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.2f} ppm")

CBW127   lat=51.970 lon=4.926  intake_height= 127 m  n=8549  mean=432.71 ppm
CBW207   lat=51.970 lon=4.926  intake_height= 207 m  n=8739  mean=430.82 ppm
CBW27    lat=51.970 lon=4.926  intake_height=  27 m  n=8526  mean=439.16 ppm
CBW67    lat=51.970 lon=4.926  intake_height=  67 m  n=8541  mean=435.18 ppm
JUE120   lat=50.910 lon=6.410  intake_height= 120 m  n=8443  mean=434.02 ppm
JUE50    lat=50.910 lon=6.410  intake_height=  50 m  n=7996  mean=436.78 ppm
JUE80    lat=50.910 lon=6.410  intake_height=  80 m  n=7992  mean=435.31 ppm
LUT0     lat=53.404 lon=6.353  intake_height=  60 m  n=8332  mean=431.81 ppm


## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [7]:
ds = xr.open_zarr("icos-fluxnet.zarr", group="_combined/fluxnet_mm", consolidated=False)

nee_nl = (
    ds["NEE"]
      .sel(ustar_threshold="VUT", nee_variant="REF")
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["NEE"].attrs.get("units", "")
for sid in nee_nl.station.values:
    da = nee_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(nee_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(nee_nl.lon.sel(station=sid)):.3f}  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.3f} {units}")

BE-Bra   lat=51.308 lon=4.520  n=12  mean=-0.735 umol m-2 s-1
BE-Maa   lat=50.980 lon=5.632  n=12  mean=-0.095 umol m-2 s-1
DE-RuS   lat=50.866 lon=6.447  n=12  mean=-0.562 umol m-2 s-1
NL-Loo   lat=52.166 lon=5.744  n=12  mean=-0.261 umol m-2 s-1


## Same query via the data-passport proxy (port 8080)

Reads the stores over HTTP through `run_proxy.py`. Each `.values` access
records chunk fetches; on session close a passport is minted.

Launch the proxy first:

```bash
python run_proxy.py --store-dir . --port 8080
```

In [ ]:
import json
from datapassport_zarr import open_zarr

OBSPACK_URL = "http://localhost:8080/icos-obspack.zarr"
FLUXNET_URL = "http://localhost:8080/icos-fluxnet.zarr"

# Same xarray expressions as above — over HTTP, with passports.

print("=== Obspack — CO2 ===")
with open_zarr(OBSPACK_URL, group="co2", verbose=True) as ds_co2:
    co2_nl = (
        ds_co2["co2"]
              .where(
                  (ds_co2.lat >= LAT_MIN) & (ds_co2.lat <= LAT_MAX) &
                  (ds_co2.lon >= LON_MIN) & (ds_co2.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time_co2=slice(T0, T1))
              .compute()                  # force chunk fetches before close
    )
    print(f"Obspack CO2 — {len(co2_nl.station)} stations × {len(co2_nl.time_co2)} timestamps")
    print(f"  mean across NL+2024 = {float(co2_nl.mean()):.2f} ppm")

obspack_passport = ds_co2._passport       # dict returned by POST /session/close

print("\n=== Fluxnet — monthly NEE ===")
with open_zarr(FLUXNET_URL, group="_combined/fluxnet_mm", verbose=True) as ds_nee:
    nee_nl = (
        ds_nee["NEE"]
              .sel(ustar_threshold="VUT", nee_variant="REF")
              .where(
                  (ds_nee.lat >= LAT_MIN) & (ds_nee.lat <= LAT_MAX) &
                  (ds_nee.lon >= LON_MIN) & (ds_nee.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time=slice(T0, T1))
              .compute()
    )
    print(f"Fluxnet NEE — {len(nee_nl.station)} stations × {len(nee_nl.time)} months")
    print(f"  mean across NL+2024 = {float(nee_nl.mean()):.3f} {ds_nee['NEE'].attrs.get('units','')}")

fluxnet_passport = ds_nee._passport

# ── The passport JSON returned by POST /session/close ─────────────────────────
print("\n──── Obspack passport ─────────────────────────────────────────────────")
print(json.dumps(obspack_passport, indent=2, default=str))

print("\n──── Fluxnet passport ─────────────────────────────────────────────────")
print(json.dumps(fluxnet_passport, indent=2, default=str))

# The wrapper also persists each passport to .passport/<timestamp>_<group>.json
# (when save_passport=True, the default).
import pathlib
saved = sorted(pathlib.Path(".passport").glob("*.json"))[-2:] if pathlib.Path(".passport").exists() else []
if saved:
    print(f"\nSaved passport files: {[str(p) for p in saved]}")